In [1]:
import pandas as pd

df = pd.read_csv("C:/Users/HP/Desktop/dataanlytics/churn analysis/data/churn_raw.csv")

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [2]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


##  Data Cleaning & Type Correction

### 1. Converting `TotalCharges` to Numeric

The `TotalCharges` column was initially stored as an **object** (string) due to the 11 blank spaces we discovered. To fix this, we use the `pd.to_numeric` function with the `errors="coerce"` argument.

**What `errors="coerce"` does:**

* It attempts to turn every string into a number.
* If it encounters a blank space or non-numeric character, it forces it to become `NaN` (Not a Number).

```python
# Executing the conversion
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

```

### 2. Post-Cleaning Audit (`df.info()`)

After running the conversion, the dataset structure has changed:

* **Type Change:** `TotalCharges` is now correctly identified as a **float64**.
* **Missing Values Found:** `TotalCharges` now shows **7,032 non-null** entries.
* **The Gap:** 7,043 (Total) - 7,032 (Non-null) = **11 missing values**.

---

### 3. Summary of Current State

| Column | Initial Type | New Type | Missing Values |
| --- | --- | --- | --- |
| **MonthlyCharges** | float64 | float64 | 0 |
| **TotalCharges** | **object** | **float64** | **11** |

###  Analytical Insight

The 11 `NaN` values we just created are exactly the same 11 rows where customers had a `tenure` of 0. Now that they are officially recognized as missing values, we can handle them mathematically.

---



In [4]:
df = df.dropna()

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7032 non-null   object 
 1   gender            7032 non-null   object 
 2   SeniorCitizen     7032 non-null   int64  
 3   Partner           7032 non-null   object 
 4   Dependents        7032 non-null   object 
 5   tenure            7032 non-null   int64  
 6   PhoneService      7032 non-null   object 
 7   MultipleLines     7032 non-null   object 
 8   InternetService   7032 non-null   object 
 9   OnlineSecurity    7032 non-null   object 
 10  OnlineBackup      7032 non-null   object 
 11  DeviceProtection  7032 non-null   object 
 12  TechSupport       7032 non-null   object 
 13  StreamingTV       7032 non-null   object 
 14  StreamingMovies   7032 non-null   object 
 15  Contract          7032 non-null   object 
 16  PaperlessBilling  7032 non-null   object 
 17  

## Final Data Purge & Verification

### 1. Handling Missing Values

Since the missing data in `TotalCharges` represents an insignificant fraction of our dataset, we have opted to **remove these rows** entirely. This ensures our model trains only on complete, verifiable financial records.

```python
# Removing the 11 rows with NaN values in TotalCharges
df = df.dropna()

```

### 2. Final Data Audit (`df.info()`)

After cleaning, our dataset is now mathematically sound:

* **Row Count:** 7,032 (Cleaned from 7,043).
* **TotalCharges:** Successfully converted to **float64**.
* **Null Count:** 0 across all 21 columns.

---

##  Data Cleaning Summary

| Action | Detail | Result |
| --- | --- | --- |
| **Type Conversion** | `TotalCharges` object → float64 | Numeric operations now possible |
| **Placeholder Fix** | Identified `" "` strings as `NaN` | Hidden missing values exposed |
| **Row Removal** | Dropped 11 rows with missing charges | 7,032 rows of high-quality data |
| **Identifier Check** | `customerID` remains for now | Will be excluded during feature selection |

---

In [6]:
categorical_cols = df.select_dtypes(include="object").columns
for col in categorical_cols:
    print(f"{col}: {df[col].unique()}")

customerID: ['7590-VHVEG' '5575-GNVDE' '3668-QPYBK' ... '4801-JZAZL' '8361-LTMKD'
 '3186-AJIEK']
gender: ['Female' 'Male']
Partner: ['Yes' 'No']
Dependents: ['No' 'Yes']
PhoneService: ['No' 'Yes']
MultipleLines: ['No phone service' 'No' 'Yes']
InternetService: ['DSL' 'Fiber optic' 'No']
OnlineSecurity: ['No' 'Yes' 'No internet service']
OnlineBackup: ['Yes' 'No' 'No internet service']
DeviceProtection: ['No' 'Yes' 'No internet service']
TechSupport: ['No' 'Yes' 'No internet service']
StreamingTV: ['No' 'Yes' 'No internet service']
StreamingMovies: ['No' 'Yes' 'No internet service']
Contract: ['Month-to-month' 'One year' 'Two year']
PaperlessBilling: ['Yes' 'No']
PaymentMethod: ['Electronic check' 'Mailed check' 'Bank transfer (automatic)'
 'Credit card (automatic)']
Churn: ['No' 'Yes']


## Feature Categorization & Mapping

Before visualizing, we must categorize our 21 columns to determine the best analytical approach for each. Our `unique()` check reveals four distinct data "buckets":

### 1. Binary Features (Simple Yes/No)

These are direct flags that we will later encode as $0$ and $1$.

* **Demographics:** `gender`, `Partner`, `Dependents`
* **Services:** `PhoneService`, `PaperlessBilling`
* **Target:** `Churn`

### 2. Multi-Category Features (The "Service" Matrix)

These columns contain 3 or more unique values. A key observation is the "No internet service" value, which acts as a secondary filter for multiple columns.

* **Contract:** `Month-to-month`, `One year`, `Two year`
* **Internet:** `DSL`, `Fiber optic`, `No`
* **Add-ons:** `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies`
* **Payment:** `Electronic check`, `Mailed check`, `Bank transfer (automatic)`, `Credit card (automatic)`

---

### 3. Numeric Features (Continuous Data)

These are our quantitative drivers. We can use these to calculate averages and correlations.

* `tenure`: How many months they've stayed.
* `MonthlyCharges`: The current bill amount.
* `TotalCharges`: The cumulative bill (now cleaned and float-typed).

### 4. The Identifier

* `customerID`: **Unique ID.** > **Note:** We keep this for now to track specific high-risk customers, but it must be dropped before training to prevent the model from learning "noise."

---

## Analytical Strategy: Handling "No Internet Service"

We noticed that for features like `OnlineSecurity` or `TechSupport`, there is a third category: **"No internet service."** * **The Problem:** If a customer doesn't have internet, they *obviously* don't have online security.

* **The Decision:** During the modeling phase, we will need to decide if we treat "No internet service" simply as "No" or as its own distinct category (since it implies the customer is a "Phone-only" user).

---

In [8]:
df.to_csv("C:/Users/HP/Desktop/dataanlytics/churn analysis/data/cleaned_dataset_for_EDA.csv", index=False)
print("data saved in \'Cleaned dataset saved.\'")

data saved in 'Cleaned dataset saved.'
